# 05 — Distillation: train your own tiny student

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenTracy/OpenTracy/blob/main/notebooks/05_distillation.ipynb)

This is the wedge — the thing a plain LLM gateway can't give you. In the previous notebook we saved ~80% by routing easy tickets to a cheaper *existing* model. Now we'll train a **custom student on our own traffic** and cut cost another 10×.

The pipeline in one paragraph: take a dataset of prompts, ask a **teacher** (GPT-4o) to generate N candidate answers per prompt, have an LLM-as-judge score them, fine-tune a tiny **student** (llama-3.2-1b) on the best ones using the BOND loss, and export a LoRA adapter you can serve behind a routing alias. 

## Before you start

> **GPU required.** In Colab: `Runtime → Change runtime type → T4 GPU` (the free tier is enough for a 1B student).
>
> **API keys:** `OPENAI_API_KEY` for the teacher. Optionally `HUGGINGFACE_HUB_TOKEN` if your student model is gated.

In [ ]:
# Verify GPU — abort if this prints an error
!nvidia-smi | head -20

## Setup

`opentracy[distill]` pulls the training extras (torch, unsloth, peft, trl, bitsandbytes). Takes 3-4 minutes on a fresh Colab runtime — go grab a coffee.

In [ ]:
!pip install "opentracy[distill,server]" -q

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

# Optional — only needed if your student model is gated on HuggingFace
# if not os.environ.get("HUGGINGFACE_HUB_TOKEN"):
#     os.environ["HUGGINGFACE_HUB_TOKEN"] = getpass("HUGGINGFACE_HUB_TOKEN: ")

## 1. Start the management API

Distillation jobs are orchestrated by the `opentracy-api` FastAPI service (normally port `:8000`). We spawn it inline for the notebook.

In [ ]:
import subprocess, time, httpx

api = subprocess.Popen(
    ["uvicorn", "opentracy.api.server:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
# Wait for it to come up
for _ in range(30):
    try:
        httpx.get("http://127.0.0.1:8000/health", timeout=2).raise_for_status()
        print("management API up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("API didn't come up — check logs")

## 2. Build a tiny dataset

In production you'd cluster your traces and promote a cluster to a dataset (see notebook 04). For this tutorial we'll upload a handful of prompts directly — the same ticket classification task from notebook 04, with ground-truth labels.

In [ ]:
import json, tempfile, pathlib

training_rows = [
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Where can I download my invoice for March?'", "response": "billing"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'My credit card was charged twice, can you refund one?'", "response": "billing"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'How do I update my billing address?'", "response": "billing"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Cancel my subscription effective today.'", "response": "billing"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Change my payment method from card to ACH.'", "response": "billing"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Getting 500 errors when calling /v1/users/me, traceback attached'", "response": "technical"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Our webhook deliveries have been silently failing for 6 hours'", "response": "technical"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'SSO with Okta broke after the SAML cert rotation'", "response": "technical"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Database connections leaking on the analytics worker'", "response": "technical"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Rate limit header says 60/min but we're being throttled at 30'", "response": "technical"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Can we get bulk export to BigQuery?'", "response": "feature_request"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Please support SAML group-based role mapping'", "response": "feature_request"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Add a Zapier integration'", "response": "feature_request"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'We need a dark mode in the dashboard'", "response": "feature_request"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Add webhooks for subscription renewals'", "response": "feature_request"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Is OpenTracy GDPR compliant?'", "response": "other"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Hi, evaluating you vs LangSmith, key differences?'", "response": "other"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'What's your uptime SLA?'", "response": "other"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Can I try it for free?'", "response": "other"},
    {"prompt": "Classify this ticket (billing/technical/feature_request/other): 'Hello'", "response": "other"},
]

dataset_path = pathlib.Path(tempfile.mkdtemp()) / "tickets.jsonl"
with open(dataset_path, "w") as f:
    for row in training_rows:
        f.write(json.dumps(row) + "\n")

print(f"wrote {dataset_path} ({len(training_rows)} rows)")

## 3. Upload the dataset and kick off a job

In [ ]:
# ---- OpenTracy platform hook-up (optional but usually what you want) ----
# By default, opentracy.completion() and opentracy.Router go DIRECT to the
# provider (OpenAI, Anthropic, etc.). That means traces, cost accounting,
# routing decisions, and the distillation loop will NOT see your calls.
#
# To make every call pass through the running OpenTracy engine so it
# shows up in the dashboard at http://<host>:3000/ — uncomment the two
# lines below BEFORE the `import opentracy` call in the next cell.
#
# Provider API keys are configured once via the UI (Settings → API Keys)
# or POST /v1/secrets/<provider>; the engine reads them from
# ~/.opentracy/secrets.json inside its container on every call.

# import os
# os.environ["OPENTRACY_ENGINE_URL"] = "http://localhost:8080"  # or your remote host


In [ ]:
from opentracy import Distiller

d = Distiller(base_url="http://127.0.0.1:8000")
print("teacher models available:", [t["id"] for t in d.teacher_models()][:5])
print("student models available:", [s["id"] for s in d.student_models()][:5])

In [ ]:
# Upload the dataset
dataset = d.upload_dataset(
    name="ticket-classification-v1",
    path=str(dataset_path),
)
print(f"dataset_id: {dataset['id']}  rows: {dataset['num_rows']}")

In [ ]:
# Estimate cost first — no job is created
est = d.estimate(
    student_model="llama-3.2-1b",
    num_prompts=len(training_rows),
    n_samples=4,
)
print(f"estimated cost:   ${est['estimated_cost']:.4f}")
print(f"sufficient:       {est['sufficient']}")

In [ ]:
# Kick it off. This runs the full 4-phase pipeline:
#   data generation → curation (BOND) → training → export
job = d.create(
    name="tickets-v1",
    dataset_id=dataset["id"],
    teacher_model="openai/gpt-4o",
    student_model="llama-3.2-1b",
    num_prompts=len(training_rows),
    n_samples=4,               # Best-of-N candidates per prompt
    training_steps=60,         # Small dataset → few steps is fine
    bond_beta=0.5,
    bond_gamma=0.1,
    export_gguf=True,
    quantization_types=["q4_k_m"],
)
print(f"job id: {job['id']}  status: {job['status']}")

## 4. Watch it train

Phases: `data_generation` (teacher calls) → `curation` (judge scoring) → `training` (fine-tune) → `export` (LoRA + GGUF). Total: ~15-25 minutes on a T4 for this tiny dataset.

In [ ]:
import time

last_phase = None
while True:
    state = d.get(job["id"])
    status = state["status"]
    phase = state.get("phase")
    progress = state.get("progress", {})
    if phase != last_phase:
        print(f"[{time.strftime('%H:%M:%S')}] phase={phase}  progress={progress}")
        last_phase = phase
    if status in ("completed", "failed", "cancelled"):
        print(f"\nFINAL: {status}")
        break
    time.sleep(20)

## 5. Pick up the artifacts

In [ ]:
artifacts = d.artifacts(job["id"])
print("adapter path:", artifacts["adapter_path"])
print("gguf paths:  ", artifacts.get("gguf_paths"))
print("metrics:     ", artifacts.get("metrics"))

What you just produced:

- `adapter_path` — the trained **LoRA adapter** (~10 MB). Load into any Transformers/PEFT pipeline.
- `gguf_paths["q4_k_m"]` — a **4-bit GGUF** file (~500 MB). Serve with llama.cpp, Ollama, vLLM, or the OpenTracy engine.

## 6. Register and swap the alias

The final step of the loop: register the distilled student in the engine's model registry, then point the alias (e.g. `model="ticket-classifier"`) at it. Your app keeps calling that alias — the model underneath just got 10× cheaper.

In [ ]:
# This registers the student in the engine's local model registry and
# serves it alongside the provider models. Your app can now call it by name.
d.register_model(
    id="ticket-classifier",
    adapter_path=artifacts["adapter_path"],
    gguf_path=artifacts["gguf_paths"].get("q4_k_m"),
)
print("registered — any request for model='ticket-classifier' routes here")

## 7. Student vs teacher — head to head

Same prompts, two models. Measure cost and agreement.

In [ ]:
import opentracy as ot

test_tickets = [
    "Please refund my accidental double charge",
    "HTTPS cert expired on auth.opentracy.ai",
    "Support Python 3.13 please",
    "Do you have a SOC 2 report?",
]

TEMPLATE = "Classify this ticket (billing/technical/feature_request/other): '{t}'"

for t in test_tickets:
    teacher = ot.completion(
        model="openai/gpt-4o",
        messages=[{"role": "user", "content": TEMPLATE.format(t=t)}],
        temperature=0, max_tokens=10,
    )
    student = ot.completion(
        model="ticket-classifier",   # the alias we just registered
        messages=[{"role": "user", "content": TEMPLATE.format(t=t)}],
        temperature=0, max_tokens=10,
        force_engine=True,
    )
    t_label = teacher.choices[0].message.content.strip().lower()
    s_label = student.choices[0].message.content.strip().lower()
    match = "✓" if t_label == s_label else "✗"
    print(f"{match} teacher={t_label:<15} student={s_label:<15}  cost: ${teacher._cost:.6f} → ${student._cost:.6f}")
    print(f"   '{t}'")

## What just happened

- ✅ Captured ticket-classification traffic as a dataset
- ✅ Ran the teacher to produce high-quality labels (BOND best-of-N)
- ✅ Fine-tuned a 1B student on those labels with LoRA
- ✅ Exported a quantized GGUF file that serves on CPU
- ✅ Registered it in the engine so `model="ticket-classifier"` now routes to the student

Every future call that would've cost `$0.001` to GPT-4o costs `$0.00002` to the student — **50× cheaper**, matching quality on *this specific task*.

Rinse and repeat for every cluster worth distilling. Over time your average cost curve compounds downward without changing any app code.

## Clean up

In [ ]:
api.terminate()
api.wait(timeout=5)
print("api shut down")

## Where to go from here

- **Run the full self-hosted stack** → [Self-host guide](https://docs.opentracy.ai/guides/self-host) (Docker Compose with ClickHouse, UI, and real trace capture)
- **Scale to production** → replace the JSONL dataset with one built from real ClickHouse traces
- **Evaluate systematically** → use `RouterEvaluator` on a holdout set to track student vs teacher agreement over time